# Option Payoff Explorer (what-if sliders)

**Family:** Interactive, animated & big data &nbsp;|&nbsp; **Library:** Altair (Vega-Lite) &nbsp;|&nbsp; **Level:** Advanced

> *Interactive, animated & big data:* What-if sliders, linked views, animation and million-point rendering. Use these when exploration or motion adds understanding that a static chart cannot.

## Design purpose

An interactive profit-and-loss chart for a single option. Sliders change the strike, implied volatility, days to expiry and option type, and the chart redraws two curves: the payoff at expiry (the hockey stick) and the Black-Scholes value today. Moving the sliders shows how time value and volatility bend the curve, which a static payoff diagram cannot show.

## When to use

- Teach how strike, volatility and time to expiry change an option's value and risk
- Compare today's mark-to-market P&L with the P&L at expiry before entering a trade
- Explore "what if implied volatility drops after earnings?" scenarios interactively

## Avoid when

- A printed report or slide (sliders do nothing there; use the static option payoff small multiples)
- Multi-leg strategies with many strikes (the parameter space gets too large for sliders)
- Exact pricing of American options or with dividends (the model here is plain Black-Scholes)

## Alternatives

- Option payoff small multiples (static, in Finance & investment)
- Volatility smile / surface

## Design tips

- The PNG shows the default state; open the HTML to interact
- Keep the y-axis label explicit ("per share") and mark the spot price and the strike
- Show both curves; the gap between them is the time value that decays as expiry approaches
- Sliders run entirely in the browser (Vega expressions include cumulativeNormal), so no Python kernel is needed

## Example

A long call or put bought today with the underlying at $100, a 3% risk-free rate and Black-Scholes pricing. Prices on the x-axis are hypothetical values of the underlying.

### Setup

In [1]:
import numpy as np
import pandas as pd
import altair as alt

PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
           "#e87ba4", "#008300", "#6250d6", "#e34948"]


@alt.theme.register("vizlib", enable=True)
def vizlib_theme():
    return alt.theme.ThemeConfig({
        "config": {
            "background": "#fcfcfb",
            "view": {"stroke": None},
            "range": {"category": PALETTE},
            "font": "Helvetica, Arial, sans-serif",
            "title": {"anchor": "start", "fontSize": 14, "color": "#0b0b0b"},
            "axis": {"gridColor": "#e6e5e1", "domainColor": "#c9c8c3", "tickColor": "#c9c8c3",
                     "labelColor": "#52514e", "titleColor": "#52514e", "titleFontWeight": "normal"},
            "legend": {"labelColor": "#52514e", "titleColor": "#52514e"},
        }
    })

### Data

In [2]:
spot, r = 100.0, 0.03  # underlying price today, risk-free rate
prices = pd.DataFrame({"S": np.round(np.linspace(50, 150, 201), 2)})
prices.head()

,S
0,50.0
1,50.5
2,51.0
3,51.5
4,52.0


### Plot

In [3]:
strike = alt.param(name="strike", value=100,
                   bind=alt.binding_range(min=70, max=130, step=1, name="Strike ($) "))
vol = alt.param(name="vol", value=0.25,
                bind=alt.binding_range(min=0.05, max=0.80, step=0.01, name="Implied volatility "))
days = alt.param(name="days", value=90,
                 bind=alt.binding_range(min=1, max=365, step=1, name="Days to expiry "))
kind = alt.param(name="kind", value="Call",
                 bind=alt.binding_radio(options=["Call", "Put"], name="Option "))

T = "(days / 365)"
xs = alt.Scale(domain=[50, 150])


def black_scholes(S):
    """Vega expression for the Black-Scholes value of the selected option at underlying price S."""
    d1 = f"((log({S} / strike) + ({r} + vol * vol / 2) * {T}) / (vol * sqrt({T})))"
    d2 = f"({d1} - vol * sqrt({T}))"
    call = f"({S} * cumulativeNormal({d1}) - strike * exp(-{r} * {T}) * cumulativeNormal({d2}))"
    put = f"({call} - {S} + strike * exp(-{r} * {T}))"  # put-call parity
    return f"(kind == 'Call' ? {call} : {put})"


premium = black_scholes(spot)
payoff = "(kind == 'Call' ? max(datum.S - strike, 0) : max(strike - datum.S, 0))"

curves = (
    alt.Chart(prices)
    .transform_calculate(premium=premium)
    .transform_calculate(expiry=f"{payoff} - datum.premium",
                         today=f"{black_scholes('datum.S')} - datum.premium")
    .transform_fold(["expiry", "today"], as_=["key", "pnl"])
    .transform_calculate(curve="datum.key == 'expiry' ? 'At expiry' : 'Today (Black-Scholes)'")
    .mark_line(strokeWidth=2.5)
    .encode(
        x=alt.X("S:Q", title="Underlying price ($)", scale=xs),
        y=alt.Y("pnl:Q", title="Profit / loss per share ($)"),
        color=alt.Color("curve:N", title=None,
                        scale=alt.Scale(domain=["At expiry", "Today (Black-Scholes)"],
                                        range=["#52514e", PALETTE[0]]),
                        legend=alt.Legend(orient="bottom", direction="horizontal")),
        tooltip=[alt.Tooltip("S:Q", title="Underlying"), "curve:N",
                 alt.Tooltip("pnl:Q", title="P&L", format="+.2f")],
    )
)
one_row = pd.DataFrame({"row": [0]})
zero = alt.Chart(one_row).mark_rule(color="#8f8e88").encode(y=alt.datum(0))
spot_rule = alt.Chart(one_row).mark_rule(color="#b8b7b1").encode(x=alt.X(datum=spot, scale=xs))
strike_rule = (alt.Chart(one_row).transform_calculate(k="strike")
               .mark_rule(color=PALETTE[1], strokeWidth=1.5).encode(x=alt.X("k:Q", scale=xs)))
strike_label = (alt.Chart(one_row).transform_calculate(k="strike", label="'Strike ' + strike")
                .mark_text(align="left", dx=5, color=PALETTE[1], fontSize=11)
                .encode(x=alt.X("k:Q", scale=xs), y=alt.value(12), text="label:N"))
readout = (
    alt.Chart(one_row)
    .transform_calculate(premium=premium)
    .transform_calculate(be="kind == 'Call' ? strike + datum.premium : strike - datum.premium")
    .transform_calculate(label="kind + ' premium today: $' + format(datum.premium, '.2f')"
                               " + '   ·   breakeven at expiry: $' + format(datum.be, '.2f')")
    .mark_text(align="left", fontSize=13, fontWeight="bold", color="#0b0b0b")
    .encode(x=alt.value(0), y=alt.value(-14), text="label:N")
)
chart = (
    alt.layer(zero, spot_rule, strike_rule, strike_label, curves, readout)
    .add_params(strike, vol, days, kind)
    .properties(width=640, height=380,
                title=alt.Title("Option P&L explorer", subtitle=[
                    "Long one option bought today with the underlying at $100 (r = 3%). Move the sliders below.", " "]))
)
chart

alt.LayerChart(...)

## How to read it

The gray line is the P&L if you hold to expiry: the premium is lost below the strike, and gains rise one-for-one above the breakeven. The blue line is the P&L if the underlying moved to that price today. It is smooth because the option still has time value. Raise the volatility or the days to expiry and the premium rises and the blue curve lifts away from the gray one. Shorten the time and the two converge.

---
Standalone script: [`option_payoff_explorer.py`](option_payoff_explorer.py) &nbsp;·&nbsp; [Catalogue](../../README.md) &nbsp;·&nbsp; [Gallery](../../gallery/index.html)